# 📰 NewsBot Intelligence System
## ITAI 2373 – Natural Language Processing Midterm Project

**Student:** Spenser Van Horne

**Course:** ITAI 2373

**Semester:** Summer 2026

---

# Project Description

The NewsBot Intelligence System is a Natural Language Processing (NLP) application designed to analyze news articles using several AI techniques. The system preprocesses news text, classifies articles into categories, extracts named entities, analyzes sentiment, performs linguistic analysis, and generates business intelligence insights.

The project demonstrates how multiple NLP methods can be integrated into one intelligent application capable of supporting media monitoring and decision-making.

---

## Objectives

This notebook demonstrates:

- Text preprocessing
- Exploratory Data Analysis
- TF-IDF Feature Engineering
- Machine Learning Classification
- Named Entity Recognition (NER)
- Part-of-Speech Tagging
- Dependency Parsing
- Sentiment Analysis
- Business Intelligence Reporting

---

## Technologies

- Python
- Pandas
- NumPy
- Matplotlib
- Scikit-learn
- spaCy
- NLTK
- TextBlob

---

## Dataset

BBC News Classification Dataset (Kaggle)

Categories:

- Business
- Entertainment
- Politics
- Sport
- Technology




In [1]:

# Installing the recommended libraries


!pip install -q kaggle
!pip install -q nltk
!pip install -q spacy
!pip install -q textblob
!pip install -q wordcloud

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:

# Now to import libraries I need


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
import spacy
import re
import os

from collections import Counter

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from textblob import TextBlob

from wordcloud import WordCloud

nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("en_core_web_sm")

print("Libraries loaded successfully.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Libraries loaded successfully.


In [13]:
!pip install kaggle

In [14]:
from google.colab import files
print("Please upload your kaggle.json file:")
uploaded = files.upload()

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json


In [15]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [32]:
!kaggle auth login

usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,benchmarks,b,config,auth}
              ...
kaggle: error: argument command: unknown parser 'auth' (choices: competitions, c, datasets, d, kernels, k, models, m, files, f, benchmarks, b, config)


In [31]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('learn-ai-bbc')

print("Path to competition files:", path)

100%|██████████| 1.85M/1.85M [00:00<00:00, 128MB/s]

Extracting files...
Path to competition files: /root/.cache/kagglehub/competitions/learn-ai-bbc


In [33]:
!kaggle competitions download -c learn-ai-bbc

100% 1.85M/1.85M [00:00<00:00, 177MB/s]



In [34]:
!unzip learn-ai-bbc.zip

Archive:  learn-ai-bbc.zip
  inflating: BBC News Sample Solution.csv  
  inflating: BBC News Test.csv       
  inflating: BBC News Train.csv      


In [35]:
!ls -la

total 6876
drwxr-xr-x 1 root root    4096 Jun 28 02:36  .
drwxr-xr-x 1 root root    4096 Jun 28 02:05  ..
-rw-r--r-- 1 root root   10369 Dec  2  2019 'BBC News Sample Solution.csv'
-rw-r--r-- 1 root root 1712432 Dec  2  2019 'BBC News Test.csv'
-rw-r--r-- 1 root root 3351206 Dec  2  2019 'BBC News Train.csv'
drwxr-xr-x 4 root root    4096 Jun  4 13:32  .config
-rw-r--r-- 1 root root      71 Jun 28 02:26  kaggle.json
-rw-r--r-- 1 root root 1936538 Dec  2  2019  learn-ai-bbc.zip
drwxr-xr-x 1 root root    4096 Jun  4 13:32  sample_data


In [36]:
import pandas as pd
import os

In [37]:
print("Available files:")
for file in os.listdir('.'):
    if file.endswith('.csv'):
        print(f"  - {file}")

Available files:
  - BBC News Train.csv
  - BBC News Test.csv
  - BBC News Sample Solution.csv


In [40]:
df = pd.read_csv('BBC News Train.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Categories: {df['Category'].unique()}")

Dataset shape: (1490, 3)
Columns: ['ArticleId', 'Text', 'Category']
Categories: ['business' 'tech' 'politics' 'sport' 'entertainment']


In [41]:
import pandas as pd
import numpy as np

In [42]:
print("Dataset Info:")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst few rows:")
print(df.head())

Dataset Info:
Shape: (1490, 3)
Columns: ['ArticleId', 'Text', 'Category']

First few rows:
   ArticleId                                               Text  Category
0       1833  worldcom ex-boss launches defence lawyers defe...  business
1        154  german business confidence slides german busin...  business
2       1101  bbc poll indicates economic gloom citizens in ...  business
3       1976  lifestyle  governs mobile choice  faster  bett...      tech
4        917  enron bosses in $168m payout eighteen former e...  business


In [44]:
# 2. Identify text and category columns
# Adjust these column names based on your dataset
text_column = 'text'  # or 'description', 'content', 'headline', etc.
category_column = 'Category'  # or 'label', 'class', etc.


In [45]:
# 3. Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())




Missing values:
ArticleId    0
Text         0
Category     0
dtype: int64


In [47]:
# 4. Remove rows with missing text or categories
text_column = 'Text'  # Correcting the column name to match 'Text' in the DataFrame
df_clean = df.dropna(subset=[text_column, category_column])

In [48]:
# 5. Check category distribution
print(f"\nCategory distribution:")
print(df_clean[category_column].value_counts())



Category distribution:
Category
sport            346
business         336
politics         274
entertainment    273
tech             261
Name: count, dtype: int64


In [49]:
# 6. Sample if dataset is too large (keep under 2000 for Colab)
if len(df_clean) > 2000:
    df_final = df_clean.sample(n=2000, random_state=42)
    print(f"\nSampled dataset to {len(df_final)} rows")
else:
    df_final = df_clean

In [50]:
# 7. Rename columns for consistency
df_final = df_final.rename(columns={
    text_column: 'content',
    category_column: 'category'
})

In [51]:
# 8. Save prepared dataset
df_final.to_csv('newsbot_dataset.csv', index=False)
print("\n✅ Dataset prepared and saved as 'newsbot_dataset.csv'")


✅ Dataset prepared and saved as 'newsbot_dataset.csv'
